# Problem 14: Dynamic Prompt Assembly

**Difficulty:** Medium | **Framework:** LangGraph

**Categories:** prompt-design, orchestration

This notebook accompanies [Agent Foundry](https://agent-foundry-theta.vercel.app/problems/dynamic-prompt-assembly).

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain langchain-openai langchain-community

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- The prompt must change based on user tier (free/pro) and user type (new/returning)
- Free-tier users must not receive pro-tier features in their prompt
- The prompt assembly logic must be clear and maintainable


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import SystemMessage, HumanMessage
from typing import TypedDict

llm = ChatOpenAI(model="gpt-4o-mini")

class AgentState(TypedDict):
    messages: list
    user_tier: str
    is_returning_user: bool

# BUG: Static prompt — doesn't adapt based on user tier or context
STATIC_PROMPT = "You are a helpful AI assistant. Answer the user's question."

def assemble_prompt(state: AgentState) -> AgentState:
    # BUG: Ignores user_tier and is_returning_user entirely
    system_msg = SystemMessage(content=STATIC_PROMPT)
    state["messages"] = [system_msg] + state["messages"]
    return state

def respond(state: AgentState) -> AgentState:
    response = llm.invoke(state["messages"])
    state["messages"].append(response)
    return state

graph = StateGraph(AgentState)
graph.add_node("assemble_prompt", assemble_prompt)
graph.add_node("respond", respond)
graph.add_edge(START, "assemble_prompt")
graph.add_edge("assemble_prompt", "respond")
graph.add_edge("respond", END)

app = graph.compile()

# Test
result = app.invoke({
    "messages": [HumanMessage(content="How do I analyze my data?")],
    "user_tier": "pro",
    "is_returning_user": True,
})
print(result["messages"][-1].content)

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. Build a function that constructs the system prompt from modular blocks based on user context
2. Use a dictionary to map tier + user-type combinations to prompt segments
3. Test all four combinations: free+new, free+returning, pro+new, pro+returning


## 7. Evaluation Criteria

Check your solution against these criteria:

- Different behavior per tier
- Recognizes returning users
- Prompt changes are correct
